# Caso Práctico 2: Análisis de Precios de Viviendas
En este ejercicio practicaremos la limpieza de un dataset inmobiliario. 
El objetivo es predecir el precio de una casa basándonos en sus metros cuadrados, año de construcción y ubicación.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="white")
print("Herramientas preparadas para el sector inmobiliario.")

Herramientas preparadas para el sector inmobiliario.


## 2. Ingesta de Datos
Cargamos el dataset de viviendas y guardamos la copia original en nuestro búnker de seguridad.

In [2]:
# Cargamos un dataset de ejemplo de casas
url = 'https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv'
df_casa = pd.read_csv(url)

# Guardamos en RAW (Nuestra regla de oro)
df_casa.to_csv('../../data/raw/viviendas_original.csv', index=False)
df_casa.head()

OSError: Cannot save file into a non-existent directory: '..\..\data\raw'

## 3. Diagnóstico de Datos
Buscamos valores nulos y analizamos las estadísticas básicas. 
¿Hay casas con 0 habitaciones? ¿Hay nulos en la ubicación?

In [ ]:
print("--- Escáner de Nulos ---")
print(df_casa.isnull().sum())

print("\n--- Análisis Estadístico (El 'Ojo del Dueño') ---")
# describe() nos da la media, el mínimo y el máximo de cada columna
display(df_casa.describe())

--- Escáner de Nulos ---
longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

--- Análisis Estadístico (El 'Ojo del Dueño') ---


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,20640.000000,20640.000000,20640.000000,20640.000000,20433.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,-119.569704,35.631861,28.639486,2635.763081,537.870553,1425.476744,499.539680,3.870671,206855.816909
std,2.003532,2.135952,12.585558,2181.615252,421.385070,1132.462122,382.329753,1.899822,115395.615874
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000
25%,-121.800000,33.930000,18.000000,1447.750000,296.000000,787.000000,280.000000,2.563400,119600.000000
50%,-118.490000,34.260000,29.000000,2127.000000,435.000000,1166.000000,409.000000,3.534800,179700.000000
75%,-118.010000,37.710000,37.000000,3148.000000,647.000000,1725.000000,605.000000,4.743250,264725.000000
max,-114.310000,41.950000,52.000000,39320.000000,6445.000000,35682.000000,6082.000000,15.000100,500001.000000


## 4. Limpieza de Datos: El problema de los Dormitorios
Hemos detectado 207 valores nulos en `total_bedrooms`. 
Aplicamos la mediana para rellenar estos huecos y asegurar que no perdemos información de esos distritos.

In [ ]:
# 1. Rellenamos los nulos con la mediana
mediana_dormitorios = df_casa['total_bedrooms'].median()
df_casa['total_bedrooms'] = df_casa['total_bedrooms'].fillna(mediana_dormitorios)

# 2. Verificamos que ya no hay nulos
print(f"Nulos en total_bedrooms tras la cirugía: {df_casa['total_bedrooms'].isnull().sum()}")

Nulos en total_bedrooms tras la cirugía: 0


## 5. Ingeniería de Características: Creación de Ratios
Para que la IA entienda mejor el valor de una zona, creamos variables relativas:
1. **Habitaciones por Hogar**: ¿Son casas grandes o apartamentos pequeños?
2. **Dormitorios por Habitación**: ¿Qué porcentaje de la casa es zona de descanso?
3. **Población por Hogar**: ¿Hay mucha densidad de gente viviendo junta?

In [ ]:
# Creamos las nuevas columnas inteligentes
df_casa['rooms_per_household'] = df_casa['total_rooms'] / df_casa['households']
df_casa['bedrooms_per_room'] = df_casa['total_bedrooms'] / df_casa['total_rooms']
df_casa['population_per_household'] = df_casa['population'] / df_casa['households']

# Miramos cómo han quedado nuestros nuevos inventos
display(df_casa[['rooms_per_household', 'bedrooms_per_room', 'population_per_household']].describe())

,rooms_per_household,bedrooms_per_room,population_per_household
count,20640.000000,20640.000000,20640.000000
mean,5.429000,0.213794,3.070655
std,2.474173,0.065248,10.386050
min,0.846154,0.037151,0.692308
25%,4.440716,0.175225,2.429741
50%,5.229129,0.203159,2.818116
75%,6.052381,0.240126,3.282261
max,141.909091,2.824675,1243.333333


## 6. Exportación del Dataset Limpio
Guardamos el resultado en la carpeta `processed`. Este dataset ahora es mucho más rico que el original porque incluye los ratios de densidad y tamaño.

In [ ]:
df_casa.to_csv('../../data/processed/viviendas_limpio.csv', index=False)
print("✅ Dataset inmobiliario guardado en data/processed/viviendas_limpio.csv")

✅ Dataset inmobiliario guardado en data/processed/viviendas_limpio.csv


## 7. Eliminación de Outliers (Valores Extremos)
Hemos detectado valores físicamente imposibles en las nuevas columnas (casas de 141 habitaciones o 1200 personas por hogar). 
Filtramos el dataset para quedarnos con datos realistas y así mejorar la precisión de los modelos futuros.

In [ ]:
# Aplicamos un filtro lógico: 
# Casas con menos de 15 habitaciones y menos de 10 personas por hogar
df_casa = df_casa[df_casa['rooms_per_household'] < 15]
df_casa = df_casa[df_casa['population_per_household'] < 10]

# Volvemos a guardar la versión definitiva "super-limpia"
df_casa.to_csv('../../data/processed/viviendas_limpio.csv', index=False)

print(f"Dataset finalizado. Filas restantes tras el filtrado: {df_casa.shape[0]}")
display(df_casa[['rooms_per_household', 'population_per_household']].describe())

Dataset finalizado. Filas restantes tras el filtrado: 20495


,rooms_per_household,population_per_household
count,20495.000000,20495.000000
mean,5.311648,2.921126
std,1.316853,0.765748
min,0.846154,0.750000
25%,4.436670,2.431617
50%,5.222586,2.820423
75%,6.033580,3.281698
max,14.851852,9.954545
